In [1]:
%load_ext autoreload
%autoreload 2

## INTRO & SETTINGS

The goal of this tutorial is to show how to construct confidence sets via **quantile regression**,
using the likelihood-based test statistics `ACORE` and `BFF`, on the "on-off" Poisson counting
experiment described in https://arxiv.org/abs/2107.03920 (Section 5.1 of the EJS paper).

A Poisson counting experiment where particle collision events are counted in the presence of both
an uncertain background process and a (new) signal process:

$$N_b \sim \text{Pois}(\nu \tau b), \qquad N_s \sim \text{Pois}(\nu b + \mu s)$$

The observable data $X = (N_s, N_b)$ contains two measurements. The parameter of interest is the
signal strength $\mu$, whereas the background scaling factor $\nu$ is a nuisance parameter. The
hyperparameters $s$, $b$, $\tau$ (expected signal count, expected background count, and the relative
measurement time between the two regions) are treated as known.

Both `ACORE` and `BFF` estimate an odds function (likelihood up to a normalizing constant) via a
probabilistic classifier, then invert a quantile-regression-based critical value to construct
confidence sets — this is the "quantile regression construction" this notebook is named for.

In [2]:
# SETTINGS

POI_DIM = 1
NUISANCE_DIM = 1
DATA_DIM = 2
BATCH_SIZE = 1  # assume we get to see only one observed sample for each "true" parameter

POI_SPACE_BOUNDS = {'low': 0, 'high': 5}       # mu
NUISANCE_SPACE_BOUNDS = {'low': 0.6, 'high': 1.4}  # nu
POI_GRID_SIZE = 1_000

CONFIDENCE_LEVEL = 0.90

B = 20_000       # simulations to train the test statistic (odds/likelihood-ratio estimator)
B_PRIME = 10_000  # simulations to train the critical-values (quantile regression) calibration model

## SIMULATE

Let's start from the simulator, which is used internally to generate the data needed to
1. estimate the test statistics;
2. estimate the critical values; and
3. diagnose the constructed confidence regions

In [3]:
import torch

from lf2i.simulator.hep import OnOff

onoff = OnOff(
    poi_grid_size=POI_GRID_SIZE,
    batch_size=BATCH_SIZE,
    poi_space_bounds=POI_SPACE_BOUNDS,
    nuisance_space_bounds=NUISANCE_SPACE_BOUNDS,
)

#### Observations

In [ ]:
# a signal-like observation (mu away from 0) and a background-only-like observation (mu equal to 0)
x_obs_signal = onoff(param=torch.tensor([[3.0, 1.0]]), batch_size=1).reshape(1, DATA_DIM)
x_obs_null = onoff(param=torch.tensor([[0.0, 1.0]]), batch_size=1).reshape(1, DATA_DIM)

x_obs_signal, x_obs_null

(tensor([[104.,  79.]]), tensor([[70., 70.]]))

## CONFIDENCE SET via ACORE

`ACORE` estimates the full odds function over $(\mu, \nu)$ jointly, then profiles out the nuisance
$\nu$ by numerical optimization at evaluation time (`param_space_bounds` tells it the box over which
to profile). This is the "textbook" nuisance-parameter treatment: nothing about $\nu$ is special-cased
here, it's simply part of the odds function's input.

In [ ]:
from lf2i.inference import LF2I
from lf2i.test_statistics import ACORE

acore = ACORE(
    estimator='gb_c',
    poi_dim=POI_DIM,
    nuisance_dim=NUISANCE_DIM,
    batch_size=BATCH_SIZE,
    data_dim=DATA_DIM,
    param_space_bounds=[
        (POI_SPACE_BOUNDS['low'], POI_SPACE_BOUNDS['high']),
        (NUISANCE_SPACE_BOUNDS['low'], NUISANCE_SPACE_BOUNDS['high']),
    ],
)
lf2i_acore = LF2I(test_statistic=acore)

In [ ]:
acore_region_signal = lf2i_acore.inference(
    x=x_obs_signal,
    evaluation_grid=onoff.poi_grid.reshape(-1, 1),
    confidence_level=CONFIDENCE_LEVEL,
    calibration_method='critical-values',
    calibration_model='cat-gb',
    simulator=onoff,
    b=B,
    b_prime=B_PRIME,
)
acore_region_null = lf2i_acore.inference(
    x=x_obs_null,
    evaluation_grid=onoff.poi_grid.reshape(-1, 1),
    confidence_level=CONFIDENCE_LEVEL,
    calibration_method='critical-values',
    calibration_model='cat-gb',
    simulator=onoff,
    b=B,
    b_prime=B_PRIME,
)

## CONFIDENCE SET via BFF (implicit nuisance marginalization)

`BFF`'s nuisance-marginalization path (numerically integrating the odds function over $\nu$ via
`scipy.integrate.nquad`) turns out to be unimplemented in the current `lf2i` codebase. Rather than
implementing that numerical-integration machinery here, we take the approach of having `BFF`
**implicitly marginalize** over $\nu$: we simulate full $(\mu, \nu)$ pairs from `OnOff` (so $X$ still
reflects the true $\nu$ variability across the training/calibration sets), but only expose $\mu$ to
`BFF`'s odds estimator — i.e. `BFF` is configured with `nuisance_dim=0` and never sees $\nu$ as an
input feature. Since $\nu$ varies across the training distribution but is not conditioned on, the
classifier learns the $\mu$-marginal-over-$\nu$ likelihood ratio automatically, rather than through
explicit integration.

This means we bypass `LF2I.inference`'s `simulator=`/`b=`/`b_prime=` convenience (which would forward
*all* parameter columns unfiltered) and instead build the training (`T`) and calibration (`T_prime`)
sets manually, dropping the $\nu$ column before handing them to `BFF`.

In [ ]:
from lf2i.test_statistics import BFF


def simulate_mu_only(size):
    mu = onoff.poi_prior.sample(sample_shape=(size,)).reshape(-1, 1)
    nu = onoff.nuisance_prior.sample(sample_shape=(size,)).reshape(-1, 1)
    samples = onoff(param=torch.hstack((mu, nu)), batch_size=BATCH_SIZE)  # (size, batch_size, data_dim)
    return mu, samples  # nu discarded -> implicit marginalization


T = simulate_mu_only(B)
T_prime = simulate_mu_only(B_PRIME)

bff = BFF(
    estimator='gb_c',
    poi_dim=POI_DIM,
    nuisance_dim=0,
    batch_size=BATCH_SIZE,
    data_dim=DATA_DIM,
)
lf2i_bff = LF2I(test_statistic=bff)

In [ ]:
bff_region_signal = lf2i_bff.inference(
    x=x_obs_signal,
    evaluation_grid=onoff.poi_grid.reshape(-1, 1),
    confidence_level=CONFIDENCE_LEVEL,
    calibration_method='critical-values',
    calibration_model='cat-gb',
    T=T,
    T_prime=T_prime,
)
bff_region_null = lf2i_bff.inference(
    x=x_obs_null,
    evaluation_grid=onoff.poi_grid.reshape(-1, 1),
    confidence_level=CONFIDENCE_LEVEL,
    calibration_method='critical-values',
    calibration_model='cat-gb',
    T=T,
    T_prime=T_prime,
)

## COMPARISON

In [ ]:
from lf2i.plot.parameter_regions import plot_parameter_regions

plot_parameter_regions(
    acore_region_signal[0], bff_region_signal[0],
    param_dim=POI_DIM,
    parameter_space_bounds=POI_SPACE_BOUNDS,
    region_names=['ACORE', 'BFF (implicit marginalization)'],
    title=f'{int(CONFIDENCE_LEVEL*100)}% confidence sets for mu -- signal-like observation',
)

In [ ]:
plot_parameter_regions(
    acore_region_null[0], bff_region_null[0],
    param_dim=POI_DIM,
    parameter_space_bounds=POI_SPACE_BOUNDS,
    region_names=['ACORE', 'BFF (implicit marginalization)'],
    title=f'{int(CONFIDENCE_LEVEL*100)}% confidence sets for mu -- background-only-like observation',
)

## EXACT COVERAGE

Since `OnOff` is cheap to simulate, we can check exact (Monte Carlo) coverage of both confidence-set
constructions across the `mu` grid. For `ACORE`, the evaluation grid must include the nuisance
dimension too (`evaluation_grid` needs `param_dim` columns) — we fix `nu` at its nominal value, 1.0,
for this diagnostic slice. For `BFF`, only `mu` is needed, since the nuisance dimension was never part
of its parameter space.

In [ ]:
nu_fixed = torch.ones(onoff.poi_grid.shape[0], 1)
acore_coverage_grid = torch.hstack((onoff.poi_grid.reshape(-1, 1), nu_fixed))

acore_grid_out, acore_coverage = lf2i_acore.coverage(
    region_type='lf2i',
    confidence_level=CONFIDENCE_LEVEL,
    calibration_method='critical-values',
    simulator=onoff,
    evaluation_grid=acore_coverage_grid,
    monte_carlo_size=1_000,
    exact=True,
)

In [ ]:
from lf2i.plot.coverage_diagnostics import coverage_probability_plot

coverage_probability_plot(
    parameters=onoff.poi_grid.reshape(-1, 1).numpy(),
    coverage_probability=acore_coverage,
    confidence_level=CONFIDENCE_LEVEL,
    param_dim=POI_DIM,
    title='ACORE -- exact coverage of mu (nu fixed at 1.0)',
)